# Shock-sensor threshold test on the completed 1D baseline

Run all cells. This uses the saved final profile of `relshock-standard`; no new Vera job is needed. Red points mark boundaries where both compression and pressure-jump thresholds are exceeded. These are raw flags, without a buffer, and do not alter the completed simulation.

This baseline used central transport with explicit diffusion. It tests detection on that profile, not the stability or accuracy of an adaptive KT/D6 solver. A smooth-wave test is still needed to assess false positives.


In [ ]:
from pathlib import Path
import sys

pencil_python = Path.home() / "Work/research/software/pencil-code/python"
sys.path.insert(0, str(pencil_python))

import pencil as pc
import numpy as np
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / ".git").exists() and (p / "runs").is_dir())
run = root / "runs/relshock-standard"
datadir = str(run / "data")
print("Pencil:", pc.__file__)
print("Data:", datadir)

In [ ]:
aver = pc.read.aver(
    datadir=datadir,
    simdir=str(run),
    plane_list=["yz"],
    avfile_list=["yzaverages_final.dat"],
    infile_list=["yzaver.in"],
    quiet=True,
)
par = pc.read.param(datadir=datadir, quiet=True)
dim = pc.read.dim(datadir=datadir)
assert dim.ny == dim.nz == 1, "These profiles assume a 1D run."
assert par.lequidist[0] and not par.lperi[0], "Expected a uniform, nonperiodic x grid."
assert not par.lshift_origin[0] and not par.lshift_origin_lower[0], "Expected an unshifted grid."
x = np.linspace(par.xyz0[0], par.xyz1[0], dim.nx)
t_final = float(aver.yz.t[-1])
vx = aver.yz.uxmx[-1]
e = aver.yz.rhomx[-1]
assert vx.shape == e.shape == x.shape
print(f"Final saved profiles: t = {t_final:g}, {len(x)} grid points")

In [ ]:
# Physical pressure: p = cs0**2 * e for this radiation-fluid setup.
p = float(par.cs0)**2 * np.asarray(e, dtype=float)
v = np.asarray(vx, dtype=float)
assert np.all(np.isfinite(p)) and np.all(p > 0)
cs = float(par.cs0)
xface = 0.5 * (x[:-1] + x[1:])
C = np.maximum(0.0, v[:-1] - v[1:]) / cs
J = np.abs(p[1:] - p[:-1]) / (p[1:] + p[:-1])
pface = 0.5 * (p[:-1] + p[1:])
thresholds = [0.01, 0.03, 0.10]
print(f"Maximum C = {C.max():.6g}; maximum J = {J.max():.6g}")
print("Raw flags only: no buffer or time persistence.")

fig_sensor, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, values, name in zip(axes, [C, J], ["Compression C", "Pressure jump J"]):
    ax.plot(xface, values)
    for threshold in thresholds:
        ax.axhline(threshold, linestyle="--", linewidth=0.8, label=f"{threshold:g}")
    ax.set_ylabel(name)
    ax.legend(title="Threshold", loc="upper left")
axes[-1].set_xlabel("Position x")
fig_sensor.tight_layout()
plt.show()


In [ ]:
fig_flags, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True, sharey=True)
print(" C*     J*     flagged / boundaries")
for row, cstar in enumerate(thresholds):
    for col, jstar in enumerate(thresholds):
        flag = (C > cstar) & (J > jstar)
        ax = axes[row, col]
        ax.plot(x, p, color="0.35", linewidth=1, label="Pressure")
        ax.scatter(xface[flag], pface[flag], color="tab:red", s=10, label="Flagged")
        ax.set_title(f"C*={cstar:g}, J*={jstar:g}: {flag.sum()} boundaries")
        print(f"{cstar:5.2f}  {jstar:5.2f}    {flag.sum():5d} / {flag.size}")
        if row == 2: ax.set_xlabel("Position x")
        if col == 0: ax.set_ylabel("Physical pressure")
axes[0, 0].legend()
fig_flags.suptitle(f"Threshold sweep on the existing baseline, t={t_final:g}")
fig_flags.tight_layout()
plt.show()


Look for flags concentrated at the sharp, compressive pressure front. The broad rarefaction should remain unflagged. If no pair detects the front, the proposed thresholds are too high for this resolved, diffused profile. Do not interpret lack of flags as absence of a shock.

Selecting a pair from this plot does not establish that it rejects smooth compressing waves or works at other resolutions. No vortical motion is present in this 1D test.
